# 01 - Data and binning: survey speed vs. image quality, by fiscal quarter and by two-week window

Builds the cached per-visit, per-night, per-quarter, per-month and per-window tables that every
version of the figure reads (`02_static_paper`, `03_static_talk`, `04_interactive`). Nothing is
plotted here except the per-period tables.

Two binnings are kept for presentation:

- **Fiscal quarters** (FY starts in October: FY26 Q1 = Oct–Dec 2025, Q2 = Jan–Mar 2026,
  Q3 = Apr–Jun 2026, Q4 = Jul–Sep 2026). A monthly table is also written for reference.
- **Two windows**: the *best two weeks of Early Ops* (the 14-night contiguous stretch of
  Oct–Dec 2025 with the most survey visits) against the *first two weeks of LSST*
  (from the season start on 20260629). Section 4.

Sources and definitions are in [README.md](README.md) and in `ohlp/reference.py`.
Nights are grouped by the integer `day_obs` (rolls over at 12:00 UTC), never by the UTC
date of an exposure.

Runs on an SDF login node (Postgres replica for ConsDB, EFD via `rubin_nights`) or in an
RSP kernel. The narrative-log weather column needs an RSP token; without one it is left
empty and the EFD WEATHER state is the only weather source.


In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings

import pandas as pd

warnings.simplefilter("ignore", FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
logging.getLogger("rubin_nights").setLevel(logging.WARNING)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

sys.path.insert(0, ".")
from ohlp import binning, data, reference as ref

In [ ]:
# Parameters
DAY_OBS_MIN = ref.DAY_OBS_EARLY_OPS_START  # 20251024, first survey night in ConsDB
DAY_OBS_MAX = 20261022  # today; months after the last night with data stay in the table as empty
REFRESH = False  # True re-queries ConsDB / EFD instead of reading ../data/*.parquet

print("Reference values drawn on the figure:")
for k, v in ref.summary().items():
    print(f"  {k:36s} {v}")

## 1. Fetch (or load) the three cached tables

- `visits`: one row per survey visit (`science_program in SCIENCE_PROGRAMS`) with the
  `rubin_nights` seeing columns `fwhm_geom`, `idiq_aos_cam`, `fwhm_500_zenith`.
- `expected`: one row per calendar night with `n_vis_expected` from
  `estimated_baseline_visit_range` and the night length between the -12 deg twilights.
- `weather`: one row per calendar night with EFD WEATHER / FAULT / DOWNTIME hours and,
  when a token is available, the narrative-log weather and fault hours.

In [ ]:
tables = data.build_cache(DAY_OBS_MIN, DAY_OBS_MAX, refresh=REFRESH)
visits, expected, weather = tables["visits"], tables["expected"], tables["weather"]
print(f"visits   {len(visits):>7d} rows, {visits.day_obs.nunique()} nights, {visits.day_obs.min()} to {visits.day_obs.max()}")
print(f"expected {len(expected):>7d} calendar nights")
print(f"weather  {len(weather):>7d} nights; EFD state known for {int(weather.weather_known.sum())}; "
      f"narrative log available: {bool(weather.narrative_log_available.any())}")
visits.head(3)

In [ ]:
# Coverage of the image-quality columns among survey visits, per month
cov = visits.groupby("month").agg(
    n_visits=("visit_id", "size"),
    n_fwhm_geom=("fwhm_geom", "count"),
    n_idiq=("idiq_aos_cam", "count"),
    n_fwhm_500=("fwhm_500_zenith", "count"),
)
cov["frac_idiq"] = (cov.n_idiq / cov.n_visits).round(3)
cov

## 2. Per-night table

One row per calendar night. `ratio = n_vis / n_vis_expected`; closed nights have
`n_vis = 0`. `is_weather_night` (`weather_hours_efd / night_hours >= WEATHER_NIGHT_FRACTION`) is
informational only: it outlines the night in the plots and is counted per period, never removed.

In [ ]:
nightly = binning.nightly_table(visits, expected, weather)
print(f"{len(nightly)} nights, {int(nightly.has_visits.sum())} with survey visits, "
      f"{int(nightly.is_weather_night.sum())} flagged as weather nights "
      f"(threshold {ref.WEATHER_NIGHT_FRACTION:.0%} of the night, EFD state from {ref.DAY_OBS_WEATHER_STATE_AVAILABLE})")
nightly.loc[nightly.is_weather_night, ["day_obs", "n_vis", "n_vis_expected", "weather_hours_efd", "night_hours", "weather_frac"]].round(2)

## 3. Per-quarter table (and per-month, for reference)

One row per period. Every calendar night counts, weather nights included: the "LSST = 1"
reference comes from the v5.1 baseline simulation, which already loses nights to its cloud and
downtime models, so removing weather here would count that loss twice. `n_nights_weather` is
kept for information. Periods with fewer than `MIN_NIGHTS_PER_MONTH` nights with visits get
`low_n = True` and are kept. Periods with no data at all (the storm gap) stay in the table with
zero visits so that the gap is explicit.

FY26 Q4 (Jul–Sep 2026) is **storm-truncated**: survey visits end on 20260713, and the remaining
nights count as zero visits. Only 20260714–20260719 carry an EFD WEATHER flag.

In [ ]:
quarterly = binning.quarterly_table(visits, nightly)
monthly = binning.monthly_table(visits, nightly)
show = [
    "label", "label_long", "n_nights_calendar", "n_nights_with_visits", "n_nights_weather",
    "n_visits", "n_vis_expected", "ratio", "ratio_q25", "ratio_q75",
    "idiq_aos_cam_med", "idiq_aos_cam_q25", "idiq_aos_cam_q75",
    "fwhm_500_zenith_med", "fwhm_geom_med", "last_day_obs_with_visits", "low_n", "is_lsst", "weather_known",
]
quarterly[show].round(3)

In [ ]:
# Monthly table, for reference
m_all = monthly[["label", "n_nights_calendar", "n_nights_with_visits", "n_visits", "n_vis_expected", "ratio", "idiq_aos_cam_med", "low_n", "is_lsst"]]
print(m_all.round(3).to_string(index=False))

## 4. Two windows: best two weeks of Early Ops vs. first two weeks of LSST

`binning.best_window` scans every contiguous `WINDOW_NIGHTS`-night window inside Oct–Dec 2025 and
ranks them by the number of survey visits (`by="n_visits"`; `by="ratio"` ranks by speed and gives
the same winner). The LSST window starts at the season start, `ref.DAY_OBS_LSST_START`
(20260629), and runs `WINDOW_NIGHTS` nights; survey visits stop on 20260713, the night before the
storm, so a 15-night window would add one more night with data.

`binning.window_table` then produces the same schema as the quarterly table (`period_kind =
"window"`, `period` = window index) so the plotting notebooks can read either file.


In [ ]:
WINDOW_NIGHTS = 14
EARLY_OPS_SEARCH = (20251001, 20251231)  # range scanned for the "best two weeks" of Early Ops

candidates = binning.best_window(nightly, *EARLY_OPS_SEARCH, n_nights=WINDOW_NIGHTS, by="n_visits")
print(f"Top {WINDOW_NIGHTS}-night windows in {EARLY_OPS_SEARCH[0]}–{EARLY_OPS_SEARCH[1]} by survey visits:")
print(candidates.head(5).round(3).to_string(index=False))
print("\n... and by speed:")
print(binning.best_window(nightly, *EARLY_OPS_SEARCH, n_nights=WINDOW_NIGHTS, by="ratio").head(3).round(3).to_string(index=False))


In [ ]:
best = candidates.iloc[0]
lsst_start = ref.DAY_OBS_LSST_START
lsst_end = int(
    nightly.loc[nightly.day_obs >= lsst_start, "day_obs"].sort_values().iloc[WINDOW_NIGHTS - 1]
)
WINDOWS = {
    "Best 2 weeks, Early Ops": (int(best.start), int(best.end)),
    "First 2 weeks, LSST": (lsst_start, lsst_end),
}
print(WINDOWS)

windows = binning.window_table(visits, nightly, WINDOWS)
windows[show].round(3)


## 5. Cross-check against a direct ConsDB count

The binned totals for two sample quarters and two sample months must equal an independent
`COUNT(*)` on `cdb_lsstcam.visit1` with the same program list. Any mismatch fails loudly.

In [ ]:
client = data.get_consdb_client()
progs = ", ".join(f"'{p}'" for p in ref.SCIENCE_PROGRAMS)
checks = []
for kind, table, keys in (("quarter", quarterly, (262, 263)), ("month", monthly, (202604, 202606))):
    for key in keys:
        lo, hi = binning.period_start_day_obs(key, kind), binning.period_end_day_obs(key, kind)
        direct = client.query(
            f"SELECT COUNT(*) AS n_visits, COUNT(DISTINCT day_obs) AS n_nights "
            f"FROM cdb_lsstcam.visit1 WHERE day_obs BETWEEN {lo} AND {hi} AND science_program IN ({progs})"
        ).iloc[0]
        binned = table.query("period == @key").iloc[0]
        checks.append({
            "period": binned.label, "day_obs_range": f"{lo}-{hi}",
            "visits_direct": int(direct.n_visits), "visits_binned": int(binned.n_visits),
            "nights_direct": int(direct.n_nights), "nights_binned": int(binned.n_nights_with_visits),
        })
checks = pd.DataFrame(checks)
assert (checks.visits_direct == checks.visits_binned).all(), checks
assert (checks.nights_direct == checks.nights_binned).all(), checks
print("Binned totals match the direct ConsDB counts.")
checks

## 6. Save the binned tables

The plotting notebooks read only these files.

In [ ]:
tag = f"{DAY_OBS_MIN}_{DAY_OBS_MAX}"
nightly.to_parquet(data.DATA_DIR / f"nightly_{tag}.parquet")
quarterly.to_parquet(data.DATA_DIR / f"quarterly_{tag}.parquet")
monthly.to_parquet(data.DATA_DIR / f"monthly_{tag}.parquet")
windows.to_parquet(data.DATA_DIR / f"windows_{tag}.parquet")
quarterly[show].round(4).to_csv(data.DATA_DIR / f"quarterly_{tag}.csv", index=False)
monthly[show].round(4).to_csv(data.DATA_DIR / f"monthly_{tag}.csv", index=False)
windows[show].round(4).to_csv(data.DATA_DIR / f"windows_{tag}.csv", index=False)
sorted(p.name for p in data.DATA_DIR.glob("*"))